# 06. Context Management (컨텍스트 관리)

에이전트 실행 중 **사용자 정보(user id, 권한 등), DB 연결 객체, API 키, 설정값** 등을 도구 함수에 전달해야 할 때가 있습니다.
`RunContextWrapper`를 사용하면 이러한 데이터를 **의존성 주입(Dependency Injection)** 방식으로 전달할 수 있습니다.

### 핵심 개념

| 개념 | 설명 |
|------|------|
| **Local Context** | 도구 함수에서 접근 가능한 로컬 데이터 (LLM에 전송되지 않음) |
| **RunContextWrapper[T]** | 컨텍스트를 감싸는 래퍼 클래스 |
| **ToolContext[T]** | RunContextWrapper + 도구 메타데이터 (도구 이름, 호출 ID 등) |

### 중요: 컨텍스트는 LLM에 전송되지 않습니다!
컨텍스트 객체는 **로컬에서만** 사용됩니다. 보안이 필요한 정보(API 키, DB 연결 등)를
안전하게 도구에 전달할 수 있습니다.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import openai

Model = "gpt-5-mini"

## 1. 기본 컨텍스트 사용법

`@dataclass`로 컨텍스트 객체를 정의하고, 도구 함수에서 `RunContextWrapper`를 통해 접근합니다.

In [9]:
from dataclasses import dataclass
from agents import Agent, Runner, RunContextWrapper, function_tool

# ============================================================
# 컨텍스트 객체 정의
# - 에이전트 실행 전체에서 공유될 사용자 정보를 담는 데이터 클래스
# - @dataclass 데코레이터로 __init__, __repr__ 등을 자동 생성
# ============================================================
@dataclass
class UserContext:
    user_id: str       # 사용자 고유 식별자
    user_name: str     # 사용자 이름
    is_premium: bool   # 프리미엄 멤버십 여부
    
# ============================================================
# 도구 함수 1: 사용자 프로필 조회
# - @function_tool: LLM이 호출할 수 있는 도구로 등록
# - RunContextWrapper[UserContext]: 타입 힌트로 컨텍스트 타입을 명시
#   → 실제 컨텍스트 객체는 wrapper.context로 접근
# ============================================================
@function_tool
async def get_user_profile(wrapper: RunContextWrapper[UserContext]) -> str:
    """현재 사용자의 프로필 정보를 조회합니다."""
    
    # RunContextWrapper에서 실제 UserContext 객체 추출
    ctx = wrapper.context
    
    # is_premium 값에 따라 멤버십 등급 문자열 결정
    membership = "프리미엄" if ctx.is_premium else "일반"
    
    # 사용자 정보를 포맷팅하여 반환 (LLM이 응답에 활용)
    return f"사용자: {ctx.user_name} (ID: {ctx.user_id}), 멤버십: {membership}"

# ============================================================
# 도구 함수 2: 개인화 추천 항목 반환
# - 컨텍스트의 is_premium 값에 따라 다른 추천 콘텐츠 제공
# - LLM에게 파라미터를 요구하지 않고 컨텍스트에서 직접 정보를 가져옴
#   → 민감한 사용자 정보가 LLM 프롬프트에 노출되지 않는 보안상 이점
# ============================================================
@function_tool
async def get_recommendations(wrapper: RunContextWrapper[UserContext]) -> str:
    """사용자에 맞는 추천 항목을 반환합니다."""
    
    # RunContextWrapper에서 실제 UserContext 객체 추출
    ctx = wrapper.context
    
    # 멤버십 등급에 따라 차별화된 추천 콘텐츠 반환
    if ctx.is_premium:
        # 프리미엄 사용자: 고급 콘텐츠 및 1:1 서비스 추천
        return f"{ctx.user_name}님을 위한 프리미엄 추천: AI 고급 과정, 1:1 멘토링, 전용 워크숍"
    else:
        # 일반 사용자: 입문 콘텐츠 및 커뮤니티 기반 서비스 추천
        return f"{ctx.user_name}님을 위한 추천: AI 입문 과정, 온라인 세미나, 커뮤니티 스터디"

## 2. 에이전트 생성 및 실행

`Agent[T]`로 컨텍스트 타입을 지정하고, `Runner.run()`의 `context` 파라미터로 전달합니다.

In [10]:
# 컨텍스트 타입이 지정된 에이전트
agent = Agent[UserContext](
    name="개인화 도우미",
    instructions="""당신은 사용자 맞춤형 도우미입니다.
사용자의 프로필과 추천 정보를 활용하여 답변하세요.
도구를 사용하여 사용자 정보를 확인할 수 있습니다.""",
    model=Model,
    tools=[get_user_profile, get_recommendations],
)

# 프리미엄 사용자 컨텍스트
premium_user = UserContext(
    user_id="USR-001",
    user_name="김영희",
    is_premium=True,
)

# 컨텍스트를 전달하며 에이전트 실행
result = await Runner.run(
    starting_agent=agent,
    input="내 프로필 정보와 추천 과정을 알려주세요.",
    context=premium_user,  # 컨텍스트 전달
)
print(f"결과: {result.final_output}")

결과: 요청하신 정보 확인했습니다.

1) 사용자 프로필 (현재 시스템에 저장된 정보)
- 이름: 김영희  
- ID: USR-001  
- 멤버십: 프리미엄

2) 현재 추천 항목 (프리미엄 대상)
- AI 고급 과정  
- 1:1 멘토링  
- 전용 워크숍

3) 추천 과정(요약 — 시스템이 일반적으로 따르는 흐름)
- 데이터 수집: 계정 정보(멤버십 등), 사용자가 직접 제공한 선호/목표, 과거 활동(수강 기록, 클릭/조회, 참여 이력), 아이템(강좌·멘토·워크숍)의 메타데이터를 수집합니다.
- 특징 추출: 사용자와 아이템 각각에 대해 주제, 난이도, 가격, 형식(온라인/오프라인), 시간대 등 특징을 정리합니다.
- 모델링/유사성 계산:
  - 콘텐츠 기반 필터링: 사용자가 과거에 선호한 항목의 속성과 유사한 항목을 찾습니다.
  - 협업 필터링: 비슷한 행동을 보인 다른 사용자가 좋아한 항목을 추천합니다.
  - 규칙/비즈니스 필터: 멤버십 전용 혜택(프리미엄 전용 과정·워크숍 등)을 우선 노출합니다.
  - 필요 시 개인화된 랭킹 모델(머신러닝)을 통해 추천 점수를 계산합니다.
- 후보군 필터링 및 정렬: 품질 기준, 재고(좌석)·일정·중복 제거, 프로모션 조건을 적용해 최종 우선순위를 정합니다.
- 후처리·검증: 인간 큐레이터 검토나 실시간 A/B 테스트로 품질을 모니터링하고, 피드백을 반영해 모델을 주기적으로 업데이트합니다.

4) 개인정보·투명성 안내
- 어떤 데이터가 사용되었는지 궁금하시면 요청하시면 항목별로 보여드립니다(예: 가입 정보, 최근 수강 기록 등).
- 추천은 주로 멤버십(프리미엄)·과거 행동·아이템 속성을 기반으로 개인화되어 제공됩니다.
- 원치 않으시면 개인화 기능을 끄거나 특정 데이터 사용을 제한할 수 있습니다(옵트아웃/설정 변경).

원하시면
- 프로필의 세부 항목(예: 선호 분야, 학습 목표 등)을 보여드리거나 수정해 드리겠습니다.
- 추천 항목 각각에 대해 왜 추천되었는지(추천 사유)를 항목별로 자세히 설명해 드리겠습니다

## 3. 동일한 에이전트, 다른 컨텍스트

같은 에이전트를 **다른 사용자 컨텍스트**로 실행하면 개인화된 결과를 얻습니다.

In [11]:
# 일반 사용자 컨텍스트
regular_user = UserContext(
    user_id="USR-002",
    user_name="박철수",
    is_premium=False,
)

result = await Runner.run(
    starting_agent=agent,
    input="나한테 맞는 추천 과정을 알려주세요.",
    context=regular_user,  # 다른 컨텍스트 전달
)
print(f"결과: {result.final_output}")

결과: 프로필(박철수님, 멤버십: 일반)과 추천 결과(AI 입문 과정, 온라인 세미나, 커뮤니티 스터디)를 바탕으로, 박철수님에게 맞춘 추천 과정(추천 항목을 선택해서 진행하는 단계별 플랜)을 아래처럼 제안드립니다.

1) 초기 진단(1회 · 15–30분)
- 목적: 현재 수준·목표·가용 시간 파악 (예: AI 기초 이해, 취업 준비, 프로젝트 적용 등)
- 방법: 간단한 설문 또는 1:1 상담(온라인)
- 결과: 개인 학습 난이도·우선순위 정함

2) 맞춤 학습 경로 설정(1회 · 30분)
- 추천 코스 결합: AI 입문 과정(기초 이론) + 온라인 세미나(심화·실무 사례) + 커뮤니티 스터디(피드백·실습)
- 기간 예시: 총 8–12주
  - 주 1–4: AI 입문(이론, 파이썬 기초 / 주 3–5시간)
  - 주 5–8: 온라인 세미나(응용 사례, 툴 사용법 / 주 2–4시간)
  - 병행: 커뮤니티 스터디(주 1회, 프로젝트 실습·코드 리뷰)
- 목표 마일스톤: 기초 개념 습득(4주), 간단한 프로젝트 완성(8주), 포트폴리오·발표(12주)

3) 학습 자원 및 일정 제안
- 필수: AI 입문 강의, 실습 노트북(구글 콜랩 등), 커뮤니티 스터디 그룹
- 권장 주간 스케줄 예:
  - 월·수: 강의 시청 + 노트 정리(각 1–1.5시간)
  - 금: 실습/과제(2시간)
  - 일(저녁): 스터디 참여(1–2시간)
- 평가: 2주마다 퀴즈/미니 과제로 진도 점검

4) 피드백 및 조정(지속)
- 정기 체크인(2주 또는 4주 간격)으로 난이도·속도 조절
- 실습 결과 기반으로 다음 학습 내용 추천(심화/확장)

5) 커뮤니티·네트워킹 활용
- 스터디에서 코드 리뷰, 문제 해결 경험 얻기
- 세미나 후 Q&A·실무자 멘토 연결 권장

6) 멤버십 관련 제안
- 현재 멤버십: 일반 — 필요 시 심화 콘텐츠나 멘토링을 원하시면 유료 옵션(프리미엄)을 안내해 드릴 수 있습니다.

원하시면 지금 바로 초기 진단(간단 설문)부터 시작할게요. 또는 우선 어떤 목표(예:

## 4. 실전 예제: 컨텍스트에 서비스 의존성 주입

실무에서는 DB 연결, API 클라이언트 등의 **서비스 의존성**을 컨텍스트에 담아 전달합니다.

In [12]:
@dataclass
class AppContext:
    user_id: str
    user_name: str
    order_db: dict  # 간단한 예시로 딕셔너리 사용 (실제로는 DB 커넥션, API 클라이언트 등이 들어갑니다)
    settings: dict

# 주문 조회 도구 - 컨텍스트의 DB를 사용
@function_tool
async def lookup_order(wrapper: RunContextWrapper[AppContext], order_id: str) -> str:
    """주문 ID로 주문 정보를 조회합니다."""
    ctx = wrapper.context
    order = ctx.order_db.get(order_id)
    if order:
        return f"{ctx.user_name}님의 주문 [{order_id}]: {order['item']} - {order['status']}"
    return f"주문 {order_id}를 찾을 수 없습니다."

@function_tool
async def get_shipping_info(wrapper: RunContextWrapper[AppContext], order_id: str) -> str:
    """주문의 배송 정보를 조회합니다."""
    ctx = wrapper.context
    order = ctx.order_db.get(order_id)
    if order and "tracking" in order:
        return f"배송 추적번호: {order['tracking']}, 예상 도착: {order.get('eta', '미정')}"
    return "배송 정보가 아직 없습니다."

# 에이전트 생성
order_agent = Agent[AppContext](
    name="주문 관리 도우미",
    instructions="""당신은 주문 관리 도우미입니다.
사용자의 주문을 조회하고 배송 정보를 안내합니다.""",
    model=Model,
    tools=[lookup_order, get_shipping_info],
)

# 애플리케이션 컨텍스트 생성 (DB, 설정 등 포함)
app_context = AppContext(
    user_id="USR-001",
    user_name="김영희",
    order_db={
        "ORD-100": {"item": "Python 교재", "status": "배송 중", "tracking": "KR1234567890", "eta": "2025-03-01"},
        "ORD-101": {"item": "AI 키보드", "status": "준비 중"},
    },
    settings={"max_orders": 10, "currency": "KRW"},
)

result = await Runner.run(
    starting_agent=order_agent,
    input="ORD-100 주문 상태랑 배송 정보 알려주세요.",
    context=app_context,
)
print(f"결과: {result.final_output}")

결과: 확인했습니다 — ORD-100의 현재 상태와 배송 정보는 다음과 같습니다.

- 주문자/주문: 김영희님 / ORD-100  
- 상품: Python 교재  
- 주문 상태: 배송 중 (In transit)  
- 배송 추적번호: KR1234567890  
- 예상 도착일: 2025-03-01

원하시면 배송사 추적 링크를 찾아 드리거나, 배송 진행 상황을 알림으로 받아보시게 설정해 드릴게요. 다른 문의사항 있으신가요?


## 5. ToolContext로 도구 메타데이터 접근

`ToolContext`는 `RunContextWrapper`를 확장하여 **도구 자체의 메타데이터**에도 접근할 수 있습니다.
- `ctx.tool_name`: 호출된 도구 이름
- `ctx.tool_call_id`: 도구 호출 고유 ID
- `ctx.tool_arguments`: 원본 인자 문자열

In [13]:
from agents.tool_context import ToolContext

# ============================================================
# ToolContext = RunContextWrapper를 상속한 클래스
# RunContextWrapper 기본 기능 + 도구 실행 메타데이터 추가 제공
# ============================================================
@function_tool
async def debug_tool(ctx: ToolContext[UserContext], query: str) -> str:
    """디버깅용 도구 - 도구 메타데이터를 출력합니다."""
    
    # ToolContext 전용 메타데이터 출력 (RunContextWrapper에는 없는 정보)
    print(f"  [디버그] 도구 이름: {ctx.tool_name}")       # 예: "debug_tool"
    print(f"  [디버그] 호출 ID: {ctx.tool_call_id}")      # 예: "call_abc123"
    print(f"  [디버그] 원본 인자: {ctx.tool_arguments}")  # 예: '{"query": "파이썬..."}'
    
    # RunContextWrapper와 동일하게 ctx.context로 UserContext 접근
    print(f"  [디버그] 사용자: {ctx.context.user_name}")
    
    return f"'{query}'에 대한 검색 결과입니다."

# ============================================================
# Agent[UserContext]: 이 에이전트가 UserContext 타입을 사용함을 명시
# → 등록된 도구들도 동일한 UserContext 타입을 사용해야 함
# ============================================================
debug_agent = Agent[UserContext](
    name="디버그 에이전트",
    instructions="사용자의 질문에 debug_tool을 사용하여 답변하세요.",
    model=Model,
    tools=[debug_tool],  # ToolContext[UserContext]를 사용하는 도구 등록
)

# ============================================================
# Runner.run(): 에이전트 실행
#     → 내부적으로 RunContextWrapper/ToolContext에 자동으로 래핑되어 전달됨
# ============================================================
result = await Runner.run(
    starting_agent=debug_agent,
    input="파이썬 비동기 프로그래밍에 대해 알려주세요.",
    context=premium_user,  # UserContext 인스턴스 → ctx.context로 접근 가능
)

# result.final_output: 에이전트가 최종적으로 반환한 텍스트 응답
print(f"\n결과: {result.final_output}")

  [디버그] 도구 이름: debug_tool
  [디버그] 호출 ID: call_IzwxEzcTo0r2LLdCrew2vO6x
  [디버그] 원본 인자: {"query":"사용자 질문에 대한 답변: '파이썬 비동기 프로그래밍에 대해 알려주세요.'"}
  [디버그] 사용자: 김영희

결과: 아래는 파이썬 비동기(asynchronous) 프로그래밍의 핵심 개념, 사용법, 예제, 주의사항과 모범 사례를 한국어로 정리한 내용입니다.

1) 비동기 프로그래밍이란?
- 비동기 프로그래밍은 I/O(입출력) 같은 대기 시간이 긴 작업을 기다리는 동안 프로그램이 다른 작업을 진행하도록 하는 방식입니다.
- 병렬(parallelism, 여러 CPU 코어 사용)과는 다르며, 비동기는 주로 단일 스레드에서 여러 작업의 실행 시점을 효율적으로 관리해 동시성(concurrency)을 높입니다.

2) 파이썬에서의 기본 도구: asyncio
- asyncio는 표준 라이브러리로 제공되는 비동기 I/O 프레임워크입니다.
- 핵심 요소:
  - event loop: 비동기 작업을 스케줄하고 실행하는 루프. asyncio.run()으로 진입하거나 asyncio.get_event_loop()로 접근.
  - coroutine: async def로 정의된 비동기 함수. 호출 시 코루틴 객체가 반환되고 await로 실행을 일시 중단/재개.
  - Task: 코루틴을 스케줄해 백그라운드에서 실행하게 하는 객체 (asyncio.create_task, asyncio.ensure_future).
  - Future: 결과를 나중에 가질 수 있는 객체(Task는 Future의 하위 개념).
  - 고수준 함수: asyncio.gather, asyncio.wait, asyncio.as_completed 등으로 여러 코루틴을 조합.

3) 기본 예제
- 간단한 코루틴과 실행:
  - async def say(name): await asyncio.sleep(1); print(f"Hello {name}")
  -

### 정리

| 개념 | 코드 | 설명 |
|------|------|------|
| 컨텍스트 정의 | `@dataclass class MyContext` | 전달할 데이터/의존성 정의 |
| 에이전트 타입 | `Agent[MyContext]` | 컨텍스트 타입 지정 |
| 도구에서 접근 | `wrapper: RunContextWrapper[MyContext]` | 도구 함수의 첫 번째 인자 |
| 실행 시 전달 | `Runner.run(..., context=my_ctx)` | context 파라미터로 전달 |
| 도구 메타데이터 | `ctx: ToolContext[MyContext]` | 도구 이름, 호출 ID 등 추가 접근 |

**활용 사례:**
- 사용자별 개인화 (프로필, 권한, 설정)
- DB 연결/API 클라이언트 주입
- 로깅/감사 추적 (audit trail)
- 요청별 설정값 전달